In [1]:
import mne
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import Pipeline
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import mne
import numpy as np
from scipy.stats import iqr
import warnings
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from utils.train_loso import train_loso_riemannian_gnn
from utils.test_loop import evaluate_riemannian_gnn
import torch 


mne.set_log_level('ERROR')   # silence MNE

warnings.filterwarnings("ignore")  # silence warnings

In [2]:
latest_channel_list = [
    # Left sensorimotor area channels
    'E29', 'E30', 'E35', 'E36', 'E41', 'E42',
    # Right sensorimotor area channels
    'E103', 'E104', 'E109', 'E110', 'E115', 'E116',
    # Mid-parietal & bilateral parietal
    'E62', 'E67', 'E72', 'E77'
 ]

new_latest = ['E24', 'E124', 'E36', 'E104', 'E47','E52', ' E60', 'E67', 'E67', 'E72', 'E77', 'E85', 'E92', 'E98', 'E62','E70', 'E75', 'E83','E58','E96','E90','E65','E69','E74','E82','E89'
              ]

bad_channels = ['E17', 'E38', 'E94', 'E113', 'E119', 'E121', 'E125', 'E128', 'E73', 'E81', 'E88', 'E43', 'E44', 'E120', 'E114','E127', 'E126',
                 'E68', 'E23', 'E3','E49','E48', "E8", "E25",
     "E56", "E63", "E99", "E107"]


                 
#bad_channels = ['E48', 'E119', 'E49', 'E113', 'E94', 'E68', 'E23', 'E3', 'E126', 'E127']



#label_dict = {'OBBA': 0, 'OBBY': 1, 'OBDO': 2, 'OBMO': 3, 'OBSI':4}
label_dict = {'IMBA': 0,'IMBY': 1,'IMSI':2 } # banana, baby, sitar
directions = ['OBBA', 'OBBY', 'OBSI']

#directions = ['OBBA', 'OBBY', 'OBDO', 'OBDO','OBSI']  # Left, Right, Up, Down

In [3]:
def get_symmetric_pairs(xml_path, tol=1e-3):
    import xml.etree.ElementTree as ET
    from collections import defaultdict

    ns = {"ns": "http://www.egi.com/sensorLayout_mff"}

    bad_channel = bad_channels
    tree = ET.parse(xml_path)
    root = tree.getroot()

    sensors = []
    for s in root.findall(".//ns:sensor", ns):
        num = int(s.find("ns:number", ns).text)

        # skip unwanted electrodes
        if num > 128 or f"E{num}" in bad_channel:
            continue

        x = float(s.find("ns:x", ns).text)
        y = float(s.find("ns:y", ns).text)
        sensors.append((num, x, y))

    midline = []
    groups = defaultdict(list)

    for num, x, y in sensors:
        if abs(x) < tol:
            midline.append('E'+str(num))
        else:
            key = (round(abs(x), 3), round(abs(y), 3))
            groups[key].append((num, x, y))

    pairs = []

    for electrodes in groups.values():
        used = set()
        for i in range(len(electrodes)):
            if i in used:
                continue
            for j in range(i + 1, len(electrodes)):
                if j in used:
                    continue

                n1, x1, y1 = electrodes[i]
                n2, x2, y2 = electrodes[j]

                if abs(x1 + x2) < tol and abs(y1 - y2) < tol:
                    pairs.append(('E'+str(n1), 'E'+str(n2)))
                    used.add(i)
                    used.add(j)
                    break

    return pairs, midline

# Usage
xml_path = "/Users/kavinfidel/Desktop/GNN+CNS+Hopf/CNS_Lab/VM_EEG/data_2/S2_/VI_S2_S1_B2_20251111_040016.mff/sensorLayout.xml"
egi_pairs, midline_sensors = get_symmetric_pairs(xml_path)

In [4]:
channel_tuple = (new_latest, bad_channels)

In [5]:

class preprocessing_pipeline:
    def __init__(self, filename, *channel_tuple, 
                 l_freq=1.0, h_freq=48.0, notch_freq=50.0, fs=500.0, time_window=1.0,
                 apply_ica=False, remove_muscle=False,
                 eog_vertical_chs=('E14', 'E21'), eog_horizontal_chs=('E1', 'E32')):
        
        self.filename = filename
        self.l_freq = l_freq
        self.h_freq = h_freq
        self.notch_freq = notch_freq
        self.time_window = time_window
        self.fs = fs
        self.active_channels = channel_tuple[0]
        self.bad_channels = channel_tuple[1]
        self.apply_ica = apply_ica
        self.remove_muscle = remove_muscle
        self.eog_vertical_chs = list(eog_vertical_chs)
        self.eog_horizontal_chs = list(eog_horizontal_chs)
        self.ica = None  # Store for inspection later

        self.raw = self.file_process()
        self.annotations = self.raw.annotations

    def file_process(self):
        raw = mne.io.read_raw_egi(self.filename, preload=True)
        
        if 'VREF' in raw.ch_names:
            raw.drop_channels(['VREF'])


        
        raw.pick('eeg')

        if self.bad_channels:
            raw.drop_channels([ch for ch in self.bad_channels if ch in raw.ch_names])

        current_ch_names = raw.ch_names

        # Filter BEFORE ICA (ICA needs broadband signal to detect artifacts)
        # Use 1Hz high-pass for ICA fitting even if analysis band is higher
        raw.notch_filter(freqs=self.notch_freq, picks='eeg', verbose=False, pad='edge')
        raw.filter(l_freq=1.0, h_freq=self.h_freq, picks='eeg', verbose=False, pad='edge')

        if self.apply_ica:
            raw = self._run_ica(raw)

        # Apply analysis band-pass AFTER ICA (if l_freq > 1.0)
        if self.l_freq > 1.0:
            raw.filter(l_freq=self.l_freq, h_freq=None, picks='eeg', verbose=False, pad='edge')

        # Average reference AFTER ICA
        raw.set_eeg_reference('average', projection=False)

        return raw

    def _run_ica(self, raw):
        """
        Adds EOG proxies, fits ICA, removes artifact components, 
        then strips proxy channels. Returns cleaned raw (EEG only).
        """
        # --- 1. Add EOG proxy channels temporarily ---
        eog_proxies_added = []

        vert_chs = [ch for ch in self.eog_vertical_chs if ch in raw.ch_names]
        if vert_chs:
            proxy = raw.copy().pick_channels(vert_chs).get_data().mean(axis=0)
            info = mne.create_info(['EOG_vertical'], raw.info['sfreq'], ch_types=['eog'])
            raw.add_channels([mne.io.RawArray(proxy[np.newaxis, :], info)], force_update_info=True)
            eog_proxies_added.append('EOG_vertical')

        horiz_chs = [ch for ch in self.eog_horizontal_chs if ch in raw.ch_names]
        if horiz_chs:
            proxy = raw.copy().pick_channels(horiz_chs).get_data().mean(axis=0)
            info = mne.create_info(['EOG_horizontal'], raw.info['sfreq'], ch_types=['eog'])
            raw.add_channels([mne.io.RawArray(proxy[np.newaxis, :], info)], force_update_info=True)
            eog_proxies_added.append('EOG_horizontal')

        # --- 2. Fit ICA on EEG channels only (not proxies) ---
        eeg_only = raw.copy().pick_types(eeg=True)
        rank = mne.compute_rank(eeg_only, tol=1e-6, tol_kind='relative')
        n_components = min(25, rank['eeg'])

        print(f"\n🔧 Fitting ICA with {n_components} components on {len(eeg_only.ch_names)} EEG channels...")
        ica = mne.preprocessing.ICA(
            n_components=n_components, 
            random_state=42,
            method='fastica', 
            max_iter=500
        )
        ica.fit(eeg_only)

        # --- 3. Detect bad components ---
        bad_components = []

        if 'EOG_vertical' in raw.ch_names:
            idx, _ = ica.find_bads_eog(raw, ch_name='EOG_vertical', threshold=2.5)
            print(f"  Vertical EOG (blinks): {idx}")
            bad_components.extend(idx)

        if 'EOG_horizontal' in raw.ch_names:
            idx, _ = ica.find_bads_eog(raw, ch_name='EOG_horizontal', threshold=2.5)
            print(f"  Horizontal EOG (saccades): {idx}")
            bad_components.extend(idx)

        if self.remove_muscle:
            try:
                idx, _ = ica.find_bads_muscle(raw, threshold=0.2)
                print(f"  Muscle artifacts: {idx}")
                bad_components.extend(idx)
            except Exception as e:
                print(f"  Muscle detection skipped: {e}")

        ica.exclude = sorted(set(bad_components))
        print(f"\n  Excluding {len(ica.exclude)}/{n_components} components: {ica.exclude}")
        self.ica = ica  # Save for later inspection

        # --- 4. Apply ICA to EEG-only copy, then re-attach annotations ---
        # Apply only to EEG channels (proxy channels are NOT passed to apply)
        raw_eeg_clean = ica.apply(eeg_only)  # operates on the eeg-only copy
        
        # Restore annotations (crop/copy loses them)
        raw_eeg_clean.set_annotations(raw.annotations)

        print(f"  ✅ ICA done. Final channel count: {len(raw_eeg_clean.ch_names)}")
        return raw_eeg_clean  # Pure EEG, proxies never re-added

    def baseline_stats(self):
        """Extract baseline statistics."""

        try:
            tmin = [ann['onset'] for ann in self.annotations if ann['description'] == 'BLCS'][0]
            tmax = [ann['onset'] for ann in self.annotations if ann['description'] == 'BLCE'][0]

            if not tmin or not tmax:
                tmin = [ann['onset'] for ann in self.annotations if ann['description'] == 'BSST'][0]
                tmax = [ann['onset'] for ann in self.annotations if ann['description'] == 'BSEN'][0]
            
            baseline_raw = self.raw.copy().crop(tmin = tmin, tmax = tmax)
            baseline_data = baseline_raw.get_data(picks = 'eeg')

            median = np.median(baseline_data, axis=1, keepdims=True)
            scale = iqr(baseline_data, axis=1, keepdims=True)

            return median, scale

        except Exception as e:
            print(f"No baseline beta:{e}")
            return 0, 1
    # def baseline_stats(self):
    #     """Extract baseline statistics."""

    #     try:
    #         # tmin = [ann['onset'] for ann in self.annotations if ann['description'] == 'BLCS' or 'BSST']
    #         # tmax = [ann['onset'] for ann in self.annotations if ann['description'] == 'BLCE' or 'BSEN']

    #         tmin= ann['onset'] for ann in self.annotations if ann['description'] in ['BLCS', 'BSST']
    #         tmax = ann['onset'] for ann in self.annotations if ann['description'] in ['BLCE', 'BSEN']
    #         # if not tmin_list or not tmax_list:
    #         #     raise ValueError("No matching baseline markers found.")
    #         # tmin = float(tmin_list)
    #         # tmax = float(tmax_list)

           
    #         baseline_raw = self.raw.copy().crop(tmin = tmin, tmax = tmax)
    #         baseline_data = baseline_raw.get_data(picks = 'eeg')

    #         median = np.median(baseline_data, axis=1, keepdims=True)
    #         scale = iqr(baseline_data, axis=1, keepdims=True)

    #         scale = np.where(scale == 0, 1, scale)

    #         return median, scale

    #     except Exception as e:
    #         print(f"No baseline beta:{e}")
    #         return 0, 1

    def extracting_data(self, start_offset=0.5, end_offset=0.0, overlap_factor=0.6, normalize = True):
            # base_mean, base_std = self.baseline_stats()
            base_mean, base_std = 0,1
            
            #classes = ['BA', 'BY', 'DO', 'MO', 'SI']
            classes = ['BA', 'BY', 'SI']
            # Changed from flat list to a dictionary grouped by class
            trial_groups = {cls: [] for cls in classes} 
            current_ch_names = self.raw.ch_names

            for cls in classes:
                starts = [ann['onset'] for ann in self.annotations if ann['description'] == f'IS{cls}']
                ends   = [ann['onset'] for ann in self.annotations if ann['description'] == f'IE{cls}']

                for start, end in zip(starts, ends):
                    segment = self.raw.copy().crop(tmin=start+start_offset, tmax=end+end_offset)
                    data = segment.get_data(picks='eeg').astype(np.float32)

                    if normalize is not None:
                        data = (data - base_mean)/base_std

                    window_samples = int(self.time_window * self.fs)
                    step_samples = int(window_samples * (1-overlap_factor))
                    
                    total_samples = data.shape[1]
                    this_trial_windows = []
                    
                    for start_pt in range(0, total_samples - window_samples + 1, step_samples):
                        chunk = data[:, start_pt:start_pt + window_samples]
                        this_trial_windows.append(chunk)

                    if this_trial_windows:
                        # Store as a tuple: (Array of Windows, Label)
                        X_windows = np.stack(this_trial_windows, axis=0)
                        y_windows = np.full(X_windows.shape[0], label_dict[f'IM{cls}'])
                        trial_groups[cls].append((X_windows, y_windows))

            return trial_groups, current_ch_names



In [6]:
# # Making the dictionary for subject dirs
# total_subjects = 1
# subject_dirs = {}
# #base_dir = r'D:\\0001_AK\\KRISHNA\\001_EEG_work_recent\\extracted_files'
# base_dir = "/Users/kavinfidel/Desktop/GNN+CNS+Hopf/CNS_Lab/VM_EEG/Data/S_13"


# for i in range(1, total_subjects + 1):
#     files = []
#     for file_name in os.listdir(base_dir):
#         #m = reg.match(r'[a-zA-Z]ursor_S(\d+)_S\d+_B\d+', file_name)
#         if not file_name.startswith('.') and file_name.endswith('.mff'):
#             files.append(file_name)
#     subject_dirs[f'Subject_{i}'] = files


In [7]:
import os

# Point this to the parent "Data" directory
#base_dir = "/home/kavinfidel/projects/VM_EEG/Data"
base_dir = "/Users/kavinfidel/Desktop/GNN+CNS+Hopf/CNS_Lab/VM_EEG/Data"
subject_dirs = {}

# 1. Get all items in the Data folder
# 2. Filter for directories that start with 'S'
sub_folders = [f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f)) and f.startswith('S')]

for folder in sub_folders:
    folder_path = os.path.join(base_dir, folder)
    files = []
    
    # List all .mff files within each subject's folder
    for file_name in os.listdir(folder_path):
        if not file_name.startswith('.') and file_name.endswith('.mff'):
            files.append(file_name)
    
    # Using the actual folder name (e.g., 'S1', 'S113') as the key
    subject_dirs[folder] = files

# Verification
print(f"Found {len(subject_dirs)} subjects.")
print("Subjects identified:", list(subject_dirs.keys()))

Found 15 subjects.
Subjects identified: ['S116', 'S118', 'S5', 'S2', 'S119', 'S117', 'S3', 'S4', 'S2_', 'S1_', 'S1', 'S6', 'S115', 'S113', 'S114']


In [8]:
total_data = {}
test_data = {}


for subject, files in subject_dirs.items(): # subject is id, files are the all the files associated with a subject
    print(f"Processing {subject}")
    
    total_data[f"{subject}"] = {} #?
    test_data[f"{subject}"] = {}
    signals = [] #?
    labels = []#?
    signals_test = []
    labels_test = []
    k = 0
    for file_name in files:
        k +=1
        file_path = os.path.join(base_dir,subject, file_name) # grabbing file path, the mff file?
        
        if not file_name.endswith('.mff'):
            print(f"Skipping non-raw file: {file_name}")
            continue
        
        required_parts = ["signal1.bin", "info1.xml"]
        missing_parts = [p for p in required_parts if not os.path.exists(os.path.join(file_path, p))] # wha tis happenign here?
        if missing_parts:
            print(f"Skipping {file_name} due to parts being missing")
            continue
        
        print(f"File is intact: {file_name}\n Beginning extraction...")
        

        try:
            processor = preprocessing_pipeline(file_path, *channel_tuple)
            # trial_data is now a dict: {'BA': [(win, lab), (win, lab), (win, lab)], ...}
            trial_data, current_chn = processor.extracting_data()

  
            # For Block 1 or 3, just put everything into Training
            for cls, trials in trial_data.items():
                for x, y in trials:
                    signals.append(x)
                    labels.append(y)

        except Exception as e:
            print(f"Error processing {file_name}: {e}")
            continue

    
    total_data[f"{subject}"]['data'] = np.concatenate(signals, axis=0)
    total_data[f"{subject}"]['labels'] = np.concatenate(labels, axis=0)   

Processing S116
File is intact: VI_S6_S1_B3__20251116_110436.mff
 Beginning extraction...
File is intact: VI_SX_S2_B1_1__20260315_022327.mff
 Beginning extraction...
File is intact: VI_S6_S1_B1__20251116_104819.mff
 Beginning extraction...
File is intact: VI_SX_S2_B2_2__20260315_023302.mff
 Beginning extraction...
File is intact: VI_SX_S2_B3_3__20260315_024046.mff
 Beginning extraction...
File is intact: VI_S6_S1_B2__20251116_105637.mff
 Beginning extraction...
Processing S118
File is intact: VI_S8_S1_B1__20251119_052228.mff
 Beginning extraction...
File is intact: VI_S8_S1_B2__20251119_053014.mff
 Beginning extraction...
File is intact: VI_S8_S1_B3__20251119_053757.mff
 Beginning extraction...
Processing S5
File is intact: S5_S1_B3_1654_161025_20251016_045503.mff
 Beginning extraction...
File is intact: S5_S1_B1_1640_161025_20251016_044052.mff
 Beginning extraction...
File is intact: S5_S1_B2_1646_161025_20251016_044828.mff
 Beginning extraction...
Processing S2
File is intact: VI_S2_

In [9]:

for subject in total_data.keys():
    data = total_data[subject]['data']
    labels = total_data[subject]['labels']
    
    print(f"--- Verification for {subject} ---")
    print(f"Data Shape: {data.shape}") 
    # Expected: (Total_Windows, Channels, Samples_per_Window)
    # Example: (1500, 128, 50) 
    
    print(f"Labels Shape: {labels.shape}")
    print(f"Unique Labels: {np.unique(labels)}")
    
    
    # Check if classes are balanced (should be roughly equal)
    unique, counts = np.unique(labels, return_counts=True)
    print("Samples per class:", dict(zip(unique, counts)))
    print("-" * 30)
    break

--- Verification for S116 ---
Data Shape: (378, 100, 500)
Labels Shape: (378,)
Unique Labels: [0 1 2]
Samples per class: {0: 126, 1: 126, 2: 126}
------------------------------


In [10]:


def compute_asymmetry(X, current_ch_names = current_chn, pairs = egi_pairs, orphans = midline_sensors):
    """
    X: Data array of shape (trials, channels, time) -> e.g., (840, 100, 250)
    current_ch_names: List of strings, the names of the channels currently in X.
    pairs: List of tuples representing (Left_CH, Right_CH).
    orphans: List of strings representing midline or unpaired channels.
    """
    left_indices = []
    right_indices = []
    new_feature_names = []

    # --- 1. Find indices for Pairs ---
    for left_ch, right_ch in pairs:
        # Crucial: Only compute if BOTH channels survived preprocessing
        if left_ch in current_ch_names and right_ch in current_ch_names:
            left_indices.append(current_ch_names.index(left_ch))
            right_indices.append(current_ch_names.index(right_ch))
            new_feature_names.append(f"{left_ch}-{right_ch}")

    # Vectorized subtraction for pairs
    X_left = X[:, left_indices, :]
    X_right = X[:, right_indices, :]
    X_asym = X_left - X_right
    

    # --- 2. Find indices for Orphans / Midline ---
    orphan_indices = []
    for ch in orphans:
        if ch in current_ch_names:
            orphan_indices.append(current_ch_names.index(ch))
            new_feature_names.append(ch)
            
    # --- 3. Combine them ---
    if orphan_indices:
        X_orphans = X[:, orphan_indices, :]
        # Stack them along the channel axis (axis=1)
        X_final = np.concatenate((X_asym, X_orphans), axis=1)
    else:
        X_final = X_asym

    print(f"Reduced from {X.shape} channels to {X_final.shape} features.")
    return X_final

In [10]:
def riemannian_model_build(X_train,y_train,X_test, clf_type):
    cov_est = Covariances(estimator='oas')
    ts = TangentSpace(metric='riemann')
    scaler = StandardScaler()
    
    # step 01
    cov_train = cov_est.fit_transform(X_train)
    cov_test = cov_est.transform(X_test)
    
    # Step 02
    X_train_ts = ts.fit_transform(cov_train)
    X_test_ts = ts.transform(cov_test)
    
    # Step 03
    X_train_scaled = scaler.fit_transform(X_train_ts)
    X_test_scaled = scaler.transform(X_test_ts)
    
    if clf_type == 'logreg':
        clf = LogisticRegression(
            penalty='l2',
            solver='lbfgs',
            class_weight='balanced',
            max_iter=10000
        )
    
    elif clf_type == 'svm':
        clf =   SVC(
            kernel='linear',
            class_weight='balanced',
            C=1.0
        )

        
    elif clf_type == 'mlp':
        clf = MLPClassifier(hidden_layer_sizes=(64, 50), max_iter=1000, early_stopping=True)
    else:
        raise ValueError('No classifier model supported')
    
    clf.fit(X_train_scaled, y_train)
    y_pred = clf.predict(X_test_scaled)
    
    return y_pred        

## model performance with confusion matrix

In [13]:
def loso_model_performance(train_dict, test_dict, clf_type, class_names=['BA', 'BY', 'SI']):
    """
    Pure LOSO: Trains on all subjects in train_dict and tests on each subject in test_dict.
    """
    total_acc_results = {}

    # 1. Prepare THE Global Training Pool
    # We combine every block from every subject currently in train_dict
    print(f"Aggregating training data from {len(train_dict)} subjects...")
    
    X_train = np.concatenate([train_dict[sub]['data'] for sub in train_dict], axis=0)
    y_train = np.concatenate([train_dict[sub]['labels'] for sub in train_dict], axis=0)
    
    X_train = compute_asymmetry(X_train)
    print(f"Total Training Pool: {X_train.shape[0]} samples.")

    # 2. Iterate over the subjects you "popped" into test_dict
    for subject in test_dict.keys():
        print(f"\n--- LOSO Testing on Subject: {subject} ---")
        
        X_test = test_dict[subject]['data']
        y_test = test_dict[subject]['labels']
        X_test = compute_asymmetry(X_test)
        print(f"Test samples for {subject}: {len(X_test)}")

        try:
            # 3. Predict (Cross-Subject Transfer)
            #y_pred = riemannian_model_build(X_train, y_train, X_test, clf_type)
            y_pred = train_loso_riemannian_gnn(train_dict, test_dict, subject)
            # 4. Accuracy
            acc = accuracy_score(y_test, y_pred)
            total_acc_results[subject] = acc
            print(f"LOSO Accuracy for {subject}: {acc:.4f}")

            # 5. Plotting
            cm = confusion_matrix(y_test, y_pred, normalize='true')
            fig, ax = plt.subplots(figsize=(8, 6))
            disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
            disp.plot(cmap='YlOrRd', ax=ax, values_format='.2g') # Different color to distinguish LOSO
            plt.title(f"Pure LOSO Confusion Matrix: {subject}\n(Train: {len(train_dict)} subs, Test: {subject})")
            
            #save_path = f"/Users/kavinfidel/Desktop/GNN+CNS+Hopf/CNS_Lab/VM_EEG/confusion_matrices/loso_{clf_type}_{subject}.png"
            #plt.savefig(save_path)
            plt.show()

        except Exception as e:
            print(f"Could not run LOSO for {subject}: {e}")

    # Summary Statistics
    if total_acc_results:
        avg_acc = np.mean(list(total_acc_results.values()))
        print(f"\nAverage Pure LOSO Accuracy: {avg_acc:.4f}")
    
    return total_acc_results

In [15]:
all_subjects = list(total_data.keys())


for test_sub in all_subjects:
    print(f"\n-- Runnning LOSO: Testing on {test_sub}")

    test_data = {test_sub: total_data[test_sub]}

    train_data = {s: total_data[s] for s in all_subjects if s != test_sub}
   

    results = loso_model_performance(train_data, test_data, clf_type= "logreg")

    


-- Runnning LOSO: Testing on S116
Aggregating training data from 14 subjects...
Reduced from (3087, 100, 500) channels to (3087, 54, 500) features.
Total Training Pool: 3087 samples.

--- LOSO Testing on Subject: S116 ---
Reduced from (378, 100, 500) channels to (378, 54, 500) features.
Test samples for S116: 378
--- Testing Subject: S116 on cpu ---
Could not run LOSO for S116: 'numpy.ndarray' object has no attribute 'dim'

-- Runnning LOSO: Testing on S118
Aggregating training data from 14 subjects...
Reduced from (3276, 100, 500) channels to (3276, 54, 500) features.
Total Training Pool: 3276 samples.

--- LOSO Testing on Subject: S118 ---
Reduced from (189, 100, 500) channels to (189, 54, 500) features.
Test samples for S118: 189
--- Testing Subject: S118 on cpu ---
Could not run LOSO for S118: 'numpy.ndarray' object has no attribute 'dim'

-- Runnning LOSO: Testing on S5
Aggregating training data from 14 subjects...
Reduced from (3276, 100, 500) channels to (3276, 54, 500) feature

KeyboardInterrupt: 

In [11]:
import torch

from sklearn.model_selection import train_test_split
from utils.data import convert_to_graph_list
import numpy as np
from sklearn.pipeline import Pipeline
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from torch_geometric.data import Data
from models.RiemannianGAT import RiemannianGAT
from torch_geometric.loader import DataLoader


def train_loso_riemannian_gnn(train_dict, test_dict, test_sub, epochs=1000, batch_size=8):
    # Setup Device
    device = torch.device("cpu")
    print(f"--- Testing Subject: {test_sub} on {device} ---")

    sub_list = ['S116', 'S118', 'S119', 'S117', 'S2_', 'S1_', 'S115', 'S113', 'S114']
    sub_list.remove(test_sub)
    
    # 1. Data Prep & Validation Split
    # Since you have small data, we split the train_dict further to get a validation set
    x_all = []
    y_all = []

    for subject in sub_list:
        raw_data = train_dict[subject]['data']
        raw_labels = train_dict[subject]['labels']
        raw_data = compute_asymmetry(raw_data)
        x_all.extend(raw_data)
        y_all.extend(raw_labels)
    

    
    
    # Stratified split ensures class balance in tiny datasets
    X_train, X_val, y_train, y_val = train_test_split(
        x_all, y_all, test_size=0.2, stratify=y_all, random_state=42
    )
    X_train = np.array(X_train)
    X_val   = np.array(X_val)
    y_train = np.array(y_train)
    y_val   = np.array(y_val)


    train_graphs = convert_to_graph_list(X_train, y_train)
    val_graphs = convert_to_graph_list(X_val, y_val)
    test_graphs = convert_to_graph_list(compute_asymmetry(test_dict[test_sub]['data']), test_dict[test_sub]['labels'])

    train_loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_graphs, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_graphs, batch_size=batch_size, shuffle=False)

    # 2. Initialize Model
    model = RiemannianGAT(in_channels=54, hidden_channels=64, out_channels=3).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4) # Added weight decay for small data
    criterion = torch.nn.CrossEntropyLoss()

    best_val_loss = float('inf')
    
    # 3. Training Loop
    for epoch in range(1, epochs + 1):
        model.train()
        total_train_loss = 0
        
        for batch in train_loader:
            batch = batch.to(device) # Move data to MPS
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
            loss = criterion(out, batch.y)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()

        # 4. Validation Phase (The "Better" Part)
        model.eval()
        total_val_loss = 0
        correct = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
                total_val_loss += criterion(out, batch.y).item()
                pred = out.argmax(dim=1)
                correct += (pred == batch.y).sum().item()

        avg_val_loss = total_val_loss / len(val_loader)
        acc = correct / len(val_graphs)

        if epoch % 10 == 0:
            print(f"Epoch {epoch:03d} | Train Loss: {total_train_loss/len(train_loader):.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {acc:.2f}")
            
        # Simple Early Stopping Logic
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), f'best_model_{subject}.pth') # Optional: Save best weights

    return model, test_loader

In [12]:
def evaluate_riemannian_gnn(model, test_loader, device, class_names=['BA', 'BY', 'SI'],weight = 'best_model.pt'):
    #model = torch.load(best_model)

    if weight:
        state_dict = torch.load(weight, map_location=device)
        model.load_state_dict(state_dict)
   
    model.to(device)
    model.eval()
    all_preds = []
    all_labels = []

    print("🧐 Evaluating on test set...")
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            # Forward pass
            out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
            preds = out.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch.y.cpu().numpy())

    # --- 1. Metrics ---
    print("\n✅ Evaluation Complete.")
    print(classification_report(all_labels, all_preds, target_names=class_names[:len(np.unique(all_labels))]))

    # --- 2. Confusion Matrix Plot ---
    cm = confusion_matrix(all_labels, all_preds, normalize='true')
    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names[:len(np.unique(all_labels))])
    disp.plot(cmap='Blues', ax=ax, values_format='.2f')
    plt.title("GNN Riemannian Confusion Matrix")
    plt.show()

    return all_labels, all_preds

In [13]:
all_subjects = list(total_data.keys())
device = torch.device('cpu')

for test_sub in all_subjects:
    print(f"\n-- Runnning LOSO: Testing on {test_sub}")

    test_data = {test_sub: total_data[test_sub]}

    train_data = {s: total_data[s] for s in all_subjects if s != test_sub}

    model, test_loader = train_loso_riemannian_gnn(train_data, test_data,test_sub)

    y_true, y_pred = evaluate_riemannian_gnn(model, test_loader, device)

    


-- Runnning LOSO: Testing on S116
--- Testing Subject: S116 on cpu ---
Reduced from (189, 100, 500) channels to (189, 54, 500) features.
Reduced from (189, 100, 500) channels to (189, 54, 500) features.
Reduced from (189, 100, 500) channels to (189, 54, 500) features.
Reduced from (378, 100, 500) channels to (378, 54, 500) features.
Reduced from (378, 100, 500) channels to (378, 54, 500) features.
Reduced from (189, 100, 500) channels to (189, 54, 500) features.
Reduced from (189, 100, 500) channels to (189, 54, 500) features.
Reduced from (189, 100, 500) channels to (189, 54, 500) features.
Reduced from (378, 100, 500) channels to (378, 54, 500) features.


AttributeError: 'numpy.ndarray' object has no attribute 'dim'